# Authorized B1 Training-balanced comparison v2 — existing runtime only
Paste the two code cells into the bottom of the EXISTING completed GPU notebook, in order. Do not replace/reconnect its runtime or rerun old training. The standalone copy here is a reference: a fresh blank runtime deliberately STOPs without the existing verified development data/cache.

Owner explicitly approved this ONE comparison on2026-10-15. Same680/53,seed42,batch16,lr.001,max5epochs,patience2,frozenResNet50/BN. Only Training-derived class weights change. No new data/weights download or upload, no Test, no final approval. The first cell uploads a small CODE-ONLY ZIP, qualifies exact27generated tests, and downloads real test evidence. Second cell runs only after all27pass with no skips. A single exclusive attempt marker prevents a repeated experiment even with another run ID.

**Provisional labels; non-clinical; non-final; participant independence unverified.** No MGI0 support; scarce MGI1. No promised improvement; retain worse outcomes.


In [ ]:
# ONE new cell in the EXISTING completed Colab GPU notebook. No dataset upload.
from pathlib import Path
assert all(n in globals() for n in ('data_root','WEIGHT_RECORD','files')), 'STOP: original completed runtime/cache required; do not rerun old training'
assert Path(data_root)==Path('/content/b1_pp1_pilot/development') and Path(data_root).is_dir()
import torch,hashlib,json,zipfile,io,tempfile,sys,os,stat,importlib
assert torch.cuda.is_available(), 'STOP: compatible existing CUDA GPU required'
assert not Path('/content/drive').exists(), 'STOP: isolated runtime without Drive required'
weighted_upload=files.upload()  # ONLY vision_b1_balanced_code_v2.zip (code, tests and aggregate reference).
assert set(weighted_upload)=={'vision_b1_balanced_code_v2.zip'}
weighted_raw=weighted_upload['vision_b1_balanced_code_v2.zip']
assert hashlib.sha256(weighted_raw).hexdigest()=='698782ceaf3f387e22a9803036c02cd9188f5184680f16c439684e8b738f0221'
weighted_archive=zipfile.ZipFile(io.BytesIO(weighted_raw))
assert weighted_archive.testzip() is None
balanced_manifest=json.loads(weighted_archive.read('verification_manifest.json'))
assert len(weighted_archive.namelist())==len(balanced_manifest['files'])+1
assert set(weighted_archive.namelist())==set(balanced_manifest['files'])|{'verification_manifest.json'}
weighted_root=Path(tempfile.mkdtemp(prefix='b1_balanced_v2_',dir='/content'))
for info in weighted_archive.infolist():
    name=info.filename; relative=Path(name)
    assert not relative.is_absolute() and '..' not in relative.parts and '\\' not in name
    assert not info.is_dir() and not stat.S_ISLNK(info.external_attr>>16) and info.file_size<=1024*1024
    payload=weighted_archive.read(name)
    if name in balanced_manifest['files']: assert hashlib.sha256(payload).hexdigest()==balanced_manifest['files'][name]
    target=weighted_root/name;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(payload)
sys.path.insert(0,str(weighted_root));os.chdir(weighted_root);importlib.invalidate_caches()
os.environ['PYTHONDONTWRITEBYTECODE']='1';sys.dont_write_bytecode=True
from src.vision import exploratory_pilot_b1_balanced_v2 as balanced
from src.vision import verify_b1_balanced_runtime_v2 as balanced_verify
for name,digest in balanced.code_hashes().items(): assert digest==balanced_manifest['files']['src/vision/'+name]
print('Fresh code-only extraction:',weighted_root,'verified files:',len(balanced_manifest['files']))
balanced_summary=balanced_verify.run(weighted_root)
print(json.dumps(balanced_summary,indent=2))
balanced_runtime_zip=balanced_verify.evidence_zip(weighted_root)
print('Generated evidence SHA256:',balanced.legacy.sha256(balanced_runtime_zip))
files.download(str(balanced_runtime_zip))
assert balanced_summary['status']=='PASS', 'STOP: all27 tests must pass, zero skips/errors/failures; do not run next cell'
balanced_proof=balanced_verify.require_pass(weighted_root/'balanced_synthetic_evidence_v2')
print('ALL27 SYNTHETIC CASES PASSED. Next cell is the separately authorized ONE weighted real pilot.')


In [ ]:
# Run only AFTER the preceding exact27-case synthetic PASS. This is the ONE authorized real comparison.
assert balanced_verify.require_pass(weighted_root/'balanced_synthetic_evidence_v2')==balanced_proof
owner_v2=json.loads((weighted_root/balanced_manifest['owner_record']).read_text())
assert balanced.trainer.digest(balanced.protocol())==owner_v2['protocol_sha256']
assert balanced.trainer.digest(balanced.code_hashes())==owner_v2['code_sha256']
weight_v2=balanced.base.original_pilot.pretrained_status()
assert weight_v2['status']=='documented_cache_present' and weight_v2['sha256']==owner_v2['weights_sha256'], 'STOP: reuse official existing cache only; do not download'
from datetime import datetime,timezone
import gc,shutil
balanced_run_id='pp1_b1_balanced_v2_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
balanced_output=Path('/content/b1_pp1_pilot/results')/balanced_run_id
balanced_auth={**owner_v2,'verification_sha256':balanced_proof,'output_run_id':balanced_run_id}
balanced_auth.pop('runtime_bindings_pending',None)
gc.collect();torch.cuda.empty_cache()
print(balanced.legacy.NOTICE,flush=True)
print('Starting ONE class-weighted head-only comparison: max5epochs,patience2; old run remains untouched.',flush=True)
balanced_results=balanced.run_pilot(data_root,balanced_auth,balanced_output,weighted_root/'balanced_synthetic_evidence_v2',opt_in=True)
assert balanced_verify.snapshot(weighted_root,balanced_manifest['files'])==balanced_manifest['files']
reference=json.loads((weighted_root/balanced_manifest['reference']).read_text())
comparison={'notice':balanced.legacy.NOTICE,'unweighted_historical_reference':reference,
            'balanced_best_epoch':balanced_results['best_epoch'],'balanced_validation':balanced_results['best_validation'],
            'selection_optimism':'Repeated Validation selection; not final held-out or clinical evaluation',
            'participant_independence_verified':False,'qualified_labels_approved':False,'test_used':False}
balanced.base.bundle.write_json(balanced_output/'comparison.json',comparison)
shutil.copyfile(weighted_root/balanced_manifest['owner_record'],balanced_output/'owner_record.json')
shutil.copytree(weighted_root/'balanced_synthetic_evidence_v2',balanced_output/'synthetic_verification')
for name in ('source_hashes_before.json','source_hashes_after.json'):
    balanced.base.bundle.write_json(balanced_output/name,balanced_manifest['files'])
assert len(list((balanced_output/'gradcam').glob('*/*.png')))==9
assert len(json.loads((balanced_output/'validation_predictions_best.json').read_text())['records'])==53
for n in ('initial_head.pt','best.pt','last.pt','pilot_results.json'):assert (balanced_output/n).is_file()
result_hashes={str(p.relative_to(balanced_output)):balanced.legacy.sha256(p) for p in sorted(balanced_output.rglob('*')) if p.is_file()}
balanced.base.bundle.write_json(balanced_output/'results_file_hashes.json',result_hashes)
print(json.dumps(comparison,indent=2,allow_nan=False))
balanced_zip=shutil.make_archive('/content/'+balanced_run_id+'_results','zip',balanced_output)
print('Actual weighted results ZIP:',balanced_zip)
print('SHA256:',balanced.legacy.sha256(balanced_zip))
files.download(balanced_zip)
